# Pembersihan Dataset Emosi Ataraxia

Notebook ini membersihkan `dataset_mentah_ataraxia.csv` (17.530 baris) menjadi dataset bersih **16.000 baris** (8 kelas x 2.000), serupa dengan `dataset_clean_ataraxia.csv`.

**Masalah pada data mentah**
1. Label tidak konsisten: huruf besar/kecil campur (`Marah`, `MARAH`), spasi di depan (` netral`)
2. Label tidak valid: kosong, `bahagia`, `unknown`, `test`, `???`, `3`
3. Teks kosong / hanya spasi
4. Noise di teks: URL (`http://t.co/...`), mention (`@budi`), karakter zero-width (U+200B), tab, spasi ganda
5. Kapitalisasi campur (`AKU SEDIH`, `Hari Ini Senang`)
6. Duplikat teks

In [ ]:
import re
import unicodedata
import pandas as pd

RAW_PATH   = "dataset_mentah_ataraxia.csv"
CLEAN_PATH = "dataset_clean_ataraxia.csv"      # opsional, hanya untuk validasi
OUT_PATH   = "dataset_clean_hasil.csv"

VALID_LABELS = {"senang", "sedih", "marah", "takut", "jijik", "kaget", "cape", "netral"}

## 1. Load data mentah

In [ ]:
df = pd.read_csv(RAW_PATH, dtype=str)
print("Shape awal:", df.shape)
df.head(10)

## 2. Eksplorasi masalah

In [ ]:
print("Missing value:\n", df.isna().sum(), "\n")
print("Jumlah variasi label mentah:", df["label"].nunique(dropna=True))
df["label"].value_counts(dropna=False)

In [ ]:
teks = df["text"].fillna("")
print("Mengandung URL     :", teks.str.contains(r"https?://").sum())
print("Mengandung mention :", teks.str.contains(r"@\w+").sum())
print("Zero-width char    :", teks.str.contains("[\u200b-\u200f\u2060\ufeff]").sum())
print("Tab                :", teks.str.contains("\t").sum())
print("Spasi ganda/tepi   :", teks.str.contains(r"\s{2,}|^\s|\s$").sum())
print("Ada huruf kapital  :", teks.str.contains(r"[A-Z]").sum())
print("Teks kosong/spasi  :", (teks.str.strip() == "").sum())

## 3. Fungsi pembersihan

Urutan: normalisasi unicode -> hapus zero-width -> hapus URL -> hapus mention -> rapikan whitespace (tab, spasi ganda) -> lowercase.

Angka dan tanda baca (`!`, `.`) **dipertahankan** karena dataset clean juga mempertahankannya (mis. `rp 200.000`, `pukul 09.00`). Karakter beraksen seperti `pokémon` juga dipertahankan.

In [ ]:
def clean_text(t):
    if pd.isna(t):
        return None
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"[\u200b-\u200f\u2060\ufeff]", "", t)   # zero-width characters
    t = re.sub(r"https?://\S+", "", t)                    # URL
    t = re.sub(r"@\w+", "", t)                            # mention
    t = re.sub(r"\s+", " ", t).strip()                    # tab, newline, spasi ganda
    return t.lower()

def clean_label(l):
    if pd.isna(l):
        return None
    return l.strip().lower()

## 4. Terapkan pembersihan

In [ ]:
data = df.copy()
data["text"]  = data["text"].map(clean_text)
data["label"] = data["label"].map(clean_label)

log = {"awal": len(data)}

# a. buang label tidak valid (kosong, bahagia, unknown, test, ???, 3)
data = data[data["label"].isin(VALID_LABELS)]
log["setelah buang label tidak valid"] = len(data)

# b. buang teks kosong
data = data[data["text"].notna() & (data["text"] != "")]
log["setelah buang teks kosong"] = len(data)

# c. buang duplikat teks
data = data.drop_duplicates(subset="text", keep="first")
log["setelah hapus duplikat"] = len(data)

data = data.reset_index(drop=True)
pd.Series(log, name="jumlah baris").to_frame()

## 5. Cek hasil

In [ ]:
print("Shape akhir:", data.shape)
print("Duplikat teks:", data["text"].duplicated().sum())
print("Teks bertentangan label:", (data.groupby("text")["label"].nunique() > 1).sum())
data["label"].value_counts()

In [ ]:
data.sample(10, random_state=42)

## 6. Simpan

In [ ]:
data[["label", "text"]].to_csv(OUT_PATH, index=False)
print("Tersimpan ke", OUT_PATH, "-", len(data), "baris")

## 7. Validasi terhadap dataset clean (opsional)

Membandingkan hasil dengan `dataset_clean_ataraxia.csv`. Urutan baris di file clean diacak, jadi perbandingan dilakukan berdasarkan isi teks.

In [ ]:
try:
    ref = pd.read_csv(CLEAN_PATH, dtype=str)
    hasil = data[["label", "text"]]

    print("Jumlah baris hasil / referensi:", len(hasil), "/", len(ref))
    print("Teks hanya di hasil     :", len(set(hasil.text) - set(ref.text)))
    print("Teks hanya di referensi :", len(set(ref.text) - set(hasil.text)))

    gabung = hasil.merge(ref, on="text", suffixes=("_hasil", "_ref"))
    print("Kecocokan label         :", (gabung.label_hasil == gabung.label_ref).mean())
except FileNotFoundError:
    print("File referensi tidak ditemukan, validasi dilewati.")